# 02. Извлечение эмбеддингов XLM-R

Прогоняем все отзывы через `xlm-roberta-base` → получаем (N, 768) векторы.
Сохраняем `X_cls.npy`, `X_mean.npy`, `labels.npy`, `ratings.npy`, `meta.json`.


In [1]:
# === Colab setup (skip if running locally) ===
import os, sys

IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    # Клонируем репо (первый раз) и переходим в него
    if not os.path.exists("/content/gan-reviews"):
        !git clone https://github.com/SultanKhassenov/gan-reviews.git /content/gan-reviews
    %cd /content/gan-reviews

    # Uninstall existing torch and related packages to ensure clean installation
    !pip uninstall -y torch torchvision torchaudio --quiet

    # Install torch, torchvision, torchaudio for CUDA 12.1.
    # Let pip automatically find the latest compatible versions for cu121.
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 --quiet

    # Now install the rest of the requirements. This should ensure compatibility.
    !pip install -q -r requirements-trainer.txt

# Добавляем корень репо в sys.path, чтобы импорты работали из notebooks/
sys.path.insert(0, os.path.abspath("../"))

# Mount Google Drive для сохранения чекпоинтов (опционально)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/gan-reviews/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
else:
    CHECKPOINT_DIR = "../checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("CHECKPOINT_DIR =", CHECKPOINT_DIR)

/content/gan-reviews
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CHECKPOINT_DIR = /content/drive/MyDrive/gan-reviews/checkpoints


In [2]:
from embeddings.extract import extract_embeddings
import json, numpy as np
from pathlib import Path

INPUT = Path('data/raw/reviews.jsonl')
OUT = Path('data/embeddings')
OUT.mkdir(parents=True, exist_ok=True)

reviews = [json.loads(line) for line in INPUT.open(encoding='utf-8')]
print(f'reviews: {len(reviews)}')


reviews: 3588


In [3]:
!mkdir -p data/embeddings

In [9]:
result = extract_embeddings(
    reviews,
    model_name='sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    batch_size=32,
    max_len=128,
)

np.save(OUT / 'X_cls.npy', result['cls'])
np.save(OUT / 'X_mean.npy', result['mean'])
np.save(OUT / 'labels.npy', result['labels'])
np.save(OUT / 'ratings.npy', result['ratings'])

meta = {
    'cat_to_id': result['cat_to_id'],
    'n_reviews': len(reviews),
    'emb_dim': int(result['cls'].shape[1]),
}
(OUT / 'meta.json').write_text(json.dumps(meta, ensure_ascii=False, indent=2))
print('saved', meta)


[extract] device=cuda, model=sentence-transformers/paraphrase-multilingual-mpnet-base-v2


tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

embedding: 100%|██████████| 113/113 [00:25<00:00,  4.47it/s]


saved {'cat_to_id': {'clothing': 0, 'cosmetics': 1, 'food': 2, 'furniture': 3, 'kids': 4, 'laptops': 5, 'large_appliances': 6, 'small_appliances': 7, 'smartphones': 8, 'sports': 9}, 'n_reviews': 3588, 'emb_dim': 768}


## Проверка: казахский vs русский в эмбеддингах

Эмпирический тест из обсуждения — XLM-R понимает казахский?


In [17]:
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
model = AutoModel.from_pretrained('sentence-transformers/paraphrase-multilingual-mpnet-base-v2').eval()

@torch.no_grad()
def emb(text):
    e = tok(text, return_tensors='pt', truncation=True)
    return model(**e).last_hidden_state[:, 0, :].numpy()

pairs = [
    ('хороший товар', 'жақсы өнім'),
    ('доставка быстрая', 'жеткізу жылдам'),
    ('качество ужасное', 'сапасы өте нашар'),
    ('качество плохое', 'сапасы жаман'),
    ('продавец вежливый, и очень приятный человек', 'сатушы көп сөйлейді'),
    ('мне понравилось', 'бағасы қымбат'),
    ('подарила сыну, очень доволен', 'екі күннен кейін сынып қалды')
]
for ru, kk in pairs:
    sim = cosine_similarity(emb(ru), emb(kk))[0, 0]
    print(f'cos({ru!r}, {kk!r}) = {sim:.3f}')


cos('хороший товар', 'жақсы өнім') = 0.955
cos('доставка быстрая', 'жеткізу жылдам') = 0.946
cos('качество ужасное', 'сапасы өте нашар') = 0.760
cos('качество плохое', 'сапасы жаман') = 0.947
cos('продавец вежливый, и очень приятный человек', 'сатушы көп сөйлейді') = 0.516
cos('мне понравилось', 'бағасы қымбат') = 0.391
cos('подарила сыну, очень доволен', 'екі күннен кейін сынып қалды') = 0.498
